# Fresh PIILO M1–M4 experiments
Enable a GPU and Internet. Upload `review2_final_source.zip` as a Kaggle input (or to `/content` in Colab). This notebook acquires the exact audited PIILO release, reproduces fixed splits and runs actual experiments. It does not implement M5–M9. The final test requires a deliberate freeze after development. Runtime can be substantial; no accuracy or robustness improvement is assumed.

In [ ]:
from pathlib import Path
import os, zipfile, subprocess, sys, shutil
roots = [Path('/kaggle/input'), Path('/content')]
archives = [p for root in roots if root.exists() for p in root.rglob('review2_final_source.zip')]
work = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
if archives:
    assert len(archives) == 1, 'Upload exactly one source bundle'
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            assert (work / member.filename).resolve().is_relative_to(work.resolve())
        archive.extractall(work)
elif Path('/kaggle/input').exists():
    # Kaggle may automatically unpack an uploaded ZIP.
    configs = list(Path('/kaggle/input').rglob('review2_final/config.json'))
    assert len(configs) == 1, 'Attach the review2_final source bundle'
    shutil.copytree(configs[0].parent, work / 'review2_final', dirs_exist_ok=True)
else:
    raise FileNotFoundError('Upload review2_final_source.zip')
os.chdir(work)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'review2_final/requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable GPU before running full experiments'
print(torch.cuda.get_device_name(0))

In [ ]:
from review2_final.notebook_runner import run, run_logged
run_logged([sys.executable, '-u', '-m', 'review2_final.acquire'], name='acquire')
run('audit')
if not Path('review2_final/data/prepared/manifest.json').exists():
    run('prepare', '--proportions', '0.7', '0.1', '0.1', '0.1')
import json
manifest = json.loads(Path('review2_final/data/prepared/manifest.json').read_text())
print(manifest['splits'])
print(manifest['coverage_warnings'])

Run a small execution check before research training. It uses actual records and a pretrained backbone but one optimization step, and must not be presented as thesis evidence. Output directories are immutable; choose a new name for a repeat. The GPU notebook starts with the documented default hyperparameters; inspect `config.json` before beginning research runs.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'review2_final/tests', '-q'], check=True)
run_logged([sys.executable, '-u', '-m', 'review2_final.tools.check_tokenization'], name='tokenization_preflight')
run('smoke', '--output', 'review2_final/runs/gpu_smoke')

In [ ]:
from review2_final.common import read
from review2_final.experiments import baseline, robustness, uncertainty
config = read('review2_final/config.json')
data = Path('review2_final/data/prepared')
m2_output = Path('review2_final/runs/baseline')
m2_result = baseline(data, m2_output, config)

In [ ]:
m3_output = Path('review2_final/runs/fgsm')
m3_result = robustness(data, m2_output, m3_output, config)

In [ ]:
m4_output = Path('review2_final/runs/uncertainty')
m4_result = uncertainty(data, m3_output, m4_output, config)

Inspect the complete learning curve, per-class results, FGSM clean/attacked comparisons and MC latency/calibration table. Do not conclude improvement from selected averages alone. Street-address validation recall is undefined. Final test evaluates every predeclared training fraction and robustness candidate, plus the selected calibrated detector; it does not select a winner using test results. Set `FREEZE_AND_EVALUATE_TEST` only when development decisions are final. After an interrupted test, `--resume` can continue only the identical frozen inputs and output path.

In [ ]:
FREEZE_AND_EVALUATE_TEST = False
if FREEZE_AND_EVALUATE_TEST:
    run('final-test', '--baseline', 'review2_final/runs/baseline', '--robustness', 'review2_final/runs/fgsm', '--uncertainty', 'review2_final/runs/uncertainty', '--output', 'review2_final/runs/final_test')

Save/download the run directories and prepared split manifests before ending the GPU session. Checkpoints are large. Dataset files contain PII; publish aggregate research reports with proper attribution rather than accidentally sharing raw examples. This notebook has no cloud retrieval or generation stage.